# RAG-Adapter research notebook
See `docs/notebooks.md` before running individual experiment sections. Outputs are cleared and local paths use the configuration cell below.


In [ ]:
from __future__ import annotations
import os
from pathlib import Path
RELEASE_ROOT = Path.cwd().resolve()
if RELEASE_ROOT.name == "notebooks":
    RELEASE_ROOT = RELEASE_ROOT.parent
DATA_ROOT = str(Path(os.environ.get("RAG_DATA_ROOT", RELEASE_ROOT / "data_local")).expanduser().resolve())
WORK_ROOT = str(Path(os.environ.get("RAG_WORK_ROOT", RELEASE_ROOT / "runs")).expanduser().resolve())
BGE_MODEL = os.environ.get("RAG_BGE_MODEL", "BAAI/bge-m3")
Path(WORK_ROOT).mkdir(parents=True, exist_ok=True)
import os, json
import numpy as np
import pyarrow.parquet as pq


# 筛选Video-MME测试视频

In [ ]:
# 使用 read_parquet 加载parquet文件
import pyarrow.parquet as pq
from collections import defaultdict
import random
import numpy as np
import os 

data = pq.ParquetFile(f"{DATA_ROOT}/dataset/Video-MME/test-00000-of-00001.parquet")

table = data.read()

sd = defaultdict(list)
md = defaultdict(list)
ld = defaultdict(list)

rsd = defaultdict(list)
rmd = defaultdict(list)
rld = defaultdict(list)

# 逐行读取文件
for i in range(data.num_row_groups):
    row_group = data.read_row_group(i)
    row_group = row_group.to_pandas()
   
    for idx, row in row_group.iterrows():
        # print(row.domain)
        if row.duration == "short":
            sd[row.domain].append(row.videoID)
        elif row.duration == "medium":
            md[row.domain].append(row.videoID)
        else:  
            ld[row.domain].append(row.videoID) 

# 从short, medium, long中6个domain中分别随机抽取15个video_id
for keys in sd.keys():
    sl = list(set(sd[keys]))
    ml = list(set(md[keys]))
    ll = list(set(ld[keys]))

    # print(keys + ": " + str(len(sl)) + " " + str(len(ml)) + " " + str(len(ll)))
    random.shuffle(sl)
    random.shuffle(ml)
    random.shuffle(ll)

    rsd[keys] = sl[:5]
    rmd[keys] = ml[:5]
    rld[keys] = ll[:5]

# 保存随机选择的视频
# np.save("{WORK_ROOT}/data/rsd.npy", rsd)
# np.save("{WORK_ROOT}/data/rmd.npy", rmd)
# np.save("{WORK_ROOT}/data/rld.npy", rld)

rsd = np.load(f"{WORK_ROOT}/data/rsd_final.npy", allow_pickle=True).item()
rmd = np.load(f"{WORK_ROOT}/data/rmd_final.npy", allow_pickle=True).item()
rld = np.load(f"{WORK_ROOT}/data/rld_final.npy", allow_pickle=True).item()


# 查看随机选择的视频中有多少在已经标注的数据集中
picked = []
for keys in rsd.keys():
    picked.extend(rsd[keys])
    picked.extend(rmd[keys])
    picked.extend(rld[keys])
# print(picked)
count = 0
for root, ds, fs in os.walk(f"{DATA_ROOT}/dataset/Video-MME/videos/videos_chunked_01/data"):
    if len(ds) == 0:
        for f in fs:
            name = f.split(".")[0]
            if name in picked:
                count += 1 
                
print(count)


In [ ]:
rsd = np.load(f"{WORK_ROOT}/data/rsd.npy", allow_pickle=True).item()
rmd = np.load(f"{WORK_ROOT}/data/rmd.npy", allow_pickle=True).item()
rld = np.load(f"{WORK_ROOT}/data/rld.npy", allow_pickle=True).item()

picked = []
for keys in rsd.keys():
    picked.extend(rsd[keys])
    picked.extend(rmd[keys])
    picked.extend(rld[keys])

videos = []
for root, ds, fs in os.walk(f"{DATA_ROOT}/dataset/Video-MME/videos"):
    if len(ds) == 0:
        for f in fs:
            name = f.split(".")[0]
            videos.append(name)

for p in picked:
    if p == '1NcGHbFWBFA':
        print(p)
    if p not in videos:
        print(p)
        
print(len(videos))


# 筛选MLVU测试视频
  

In [ ]:
import json
from collections import defaultdict
import random
import numpy as np
import os 

# 已经标注视频
video_ids = []
for root, ds, fs in os.walk(f"{DATA_ROOT}/dataset/MLVU/frames"):
    if len(ds) == 0:
        video_id = root.split("/")[-1]
        video_ids.append(video_id)
for root, ds, fs in os.walk(f"{DATA_ROOT}/dataset/added_frames"):
    if len(ds) == 0:
        video_id = root.split("/")[-1]
        video_ids.append(video_id)


tasks = ['1_plotQA', '2_needle', '3_ego', '4_count', '5_order', '6_anomaly_reco', '7_topic_reasoning', '8_sub_scene', '9_summary']
unpicked = defaultdict(list)
picked = defaultdict()
picked_task = defaultdict(set)

for task in tasks:
    file = os.path.join(f"{DATA_ROOT}/dataset/MLVU", f"MLVU_json_{task}.json")
    with open(file, 'r', encoding='utf-8') as f:
        qs = json.load(f)
    for q in qs:
        video_id = q["video"].split(".")
        video_id = video_id[0]
        if video_id in video_ids:
            # picked_task[q["question_type"]].add(video_id)
            if video_id in picked:
                continue
            picked[video_id] = q["question_type"]
        unpicked[q["question_type"]].append(video_id)

print(picked_task)
# 已经标注视频种类和数量
cats = ['plotQA', 'findNeedle', 'ego', 'count', 'order', 'anomaly_reco', 'topic_reasoning', 
        'subPlot', 'summary']
cats_cnt = defaultdict()
for cat in cats:
    cats_cnt[cat] = 0
for k, v in picked.items():
    cat_index = cats.index(v)
    cats_cnt[cats[cat_index]] += 1
print(cats_cnt)

# 计算每个task需要补充的视频数量
task_to_add = defaultdict(int)
for k, v in cats_cnt.items():
    task_to_add[k] = 10 - v if v < 10 else 0
print(task_to_add)

# 从未标注的视频中随机选择
to_be_picked = defaultdict(list)
for k, v in unpicked.items():
    uni_v = list(set(v))
    random.shuffle(uni_v)
    to_be_picked[k] = uni_v[:task_to_add[k]]
for k, v in to_be_picked.items():
    print(k, v)
# np.save("{WORK_ROOT}/data/mlvu_picked.npy", to_be_picked)
# print(to_be_picked)
